> ## SOLUTION / ANSWER KEY &mdash; Lab 5.1
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-01-agent-vs-model.ipynb`](../lab-01-agent-vs-model.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.1 &mdash; A Model that Answers vs an Agent that Acts

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Call the real model twice and see it is stateless (no memory between calls)
- Write a rule that DECIDES an action toward a goal
- Wrap it in a loop that carries state and stops when the goal is met

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; A model that answers &rarr; an agent that acts](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = "/tmp/biaa-lab-05-01"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
A **language model** is a brilliant brain with no hands: give it a prompt, it returns text **once**, then
forgets. An **agent** wraps a model so it can take an **action toward a goal**, look at the result, and
**keep going** until the goal is met &mdash; carrying **state** between steps. Below you'll feel both: the
real model answering statelessly, and a tiny agent loop that remembers where it is.

In [ ]:
# A model maps text -> text, once, with no memory. An agent wraps it in a loop with a goal + state.
# (We call the REAL model in the "Run it for real" cell below.)
print("model:  prompt -> text (stateless)")
print("agent:  goal + loop + state -> keeps acting until done")

## Build it
Write `decide` (the action toward the goal) and note how `run` **carries state** (`current`) across
steps &mdash; the thing a bare model can't do.

In [ ]:
def decide(goal_target, current):
    # the agent DECIDES the next action toward the goal
    if current >= goal_target:
        return "done"
    return "increment"

def run(goal_target, max_steps=20):
    current, steps = 0, 0          # the loop CARRIES STATE across steps
    while steps < max_steps:
        if decide(goal_target, current) == "done":
            return {"value": current, "steps": steps, "stopped": "goal"}
        current += 1; steps += 1
    return {"value": current, "steps": steps, "stopped": "max_steps"}

try:
    print("agent reached", run(3))   # stateful: it remembers `current` and stops at the goal
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Call the real model twice with the same prompt and watch it answer identically with no memory &mdash; then compare to your stateful loop above.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        q = "In ONE word, greet me."
        a1 = llm_text(q)     # a REAL model call
        a2 = llm_text(q)     # the SAME prompt again
        print("model call 1:", a1)
        print("model call 2:", a2)
        print("---")
        print("Same prompt -> same reply, and call 2 has NO memory of call 1: the model is STATELESS.")
        print("Your run() loop, by contrast, carried state (`current`) across steps until the goal was met.")
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The real model returned the **same** reply both times and had **no idea** it had already answered &mdash; stateless.
- Your `run()` loop kept `current` between steps and **stopped at the goal** &mdash; that memory + loop is the seed of every agent.
- Everything else in this module &mdash; tools, the ReAct loop, memory, guardrails &mdash; hangs off that single shift.

## Your turn (open task &mdash; no grader)
Ask the real model a follow-up that needs memory &mdash; e.g. call `llm_text("what did I just ask you?")` &mdash;
and watch it have no clue. Then change `run`'s target and confirm the loop still tracks its state to the goal.
**What good looks like:** the model can't remember across calls; your loop can.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
if ollama_up():
    print("follow-up (needs memory):", llm_text("What did I just ask you?"))   # the model has no clue
else:
    print("(start Ollama to see the stateless model reply)")
print("run(target=7) ->", run(7))   # the loop still tracks its own state to the NEW goal

---
### Key takeaway
A model **answers**; an agent **acts toward a goal in a loop with state**. You just proved a real model is stateless -- now you'll give the loop tools, parsing, memory and guardrails.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>